# Control demo de vacaciones y permisos — MariaDB en Colab

Este cuaderno instala MariaDB en la máquina temporal de Colab, crea la base `rrhh_demo`, carga `sql/schema.sql` y permite consultar e ingresar registros ficticios. Al reiniciar o cerrar la sesión, vuelve a ejecutar las celdas de instalación y carga. No uses datos reales de trabajadores.

## 1. Instalar MariaDB y obtener el SQL del repositorio

Ejecuta esta celda. Si ya clonaste el repositorio en esta sesión, se reutiliza esa copia.

In [ ]:
from pathlib import Path
import subprocess
import os

repo = Path('/content/Municipalidad-Colab')
if not (repo / 'sql' / 'schema.sql').exists():
    subprocess.run(['git', 'clone', 'https://github.com/GimAlvieri/Municipalidad-Colab.git', str(repo)], check=True)

!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y mariadb-server
!service mariadb start
print('SQL del proyecto:', repo / 'sql' / 'schema.sql')

## 2. Crear la base, usuario de práctica y tablas

La contraseña `colab_demo` solo se usa dentro de esta sesión temporal de Colab para los datos ficticios. No la uses como contraseña real. Esta celda crea la base y carga el archivo SQL del repositorio.

In [ ]:
!mysql -u root -e "CREATE DATABASE IF NOT EXISTS rrhh_demo CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci; CREATE USER IF NOT EXISTS 'rrhh_app'@'localhost' IDENTIFIED BY 'colab_demo'; CREATE USER IF NOT EXISTS 'rrhh_app'@'127.0.0.1' IDENTIFIED BY 'colab_demo'; GRANT ALL PRIVILEGES ON rrhh_demo.* TO 'rrhh_app'@'localhost'; GRANT ALL PRIVILEGES ON rrhh_demo.* TO 'rrhh_app'@'127.0.0.1'; FLUSH PRIVILEGES;"
!mysql -u root rrhh_demo < /content/Municipalidad-Colab/sql/schema.sql
!mysql -u root rrhh_demo -e "SHOW TABLES; SELECT * FROM resumen_por_estado ORDER BY area, estado, tipo;"

## 3. Conectarse desde Python y ver el resumen

La conexión es a MariaDB dentro de esta misma sesión de Colab; no pide una cadena de Supabase.

In [ ]:
%pip -q install mysql-connector-python pandas ipywidgets
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(host='127.0.0.1', port=3306, user='rrhh_app',
                               password='colab_demo', database='rrhh_demo')
cur = conn.cursor(dictionary=True)
cur.execute('SELECT area, estado, tipo, cantidad FROM resumen_por_estado ORDER BY area, estado, tipo')
resumen = pd.DataFrame(cur.fetchall())
display(resumen)

if not resumen.empty:
    grafico = resumen.groupby('estado')['cantidad'].sum().sort_values(ascending=False)
    ax = grafico.plot(kind='bar', title='Solicitudes ficticias por estado',
                      xlabel='Estado', ylabel='Cantidad', color='#179b83', rot=0)
    ax.figure.tight_layout()

In [ ]:
cur.execute('''SELECT s.id, s.codigo_trabajador, a.nombre AS area, s.tipo,
                     s.fecha_inicio, s.fecha_fin, s.estado
              FROM solicitudes s JOIN areas a ON a.id = s.area_id
              ORDER BY s.fecha_inicio''')
display(pd.DataFrame(cur.fetchall()))

## 4. Registrar una solicitud ficticia

Usa un código `DEMO-...`; no escribas nombres, DNI ni datos reales.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

cur.execute('SELECT id, nombre FROM areas ORDER BY nombre')
areas = cur.fetchall()
area_widget = widgets.Dropdown(options=[(a['nombre'], a['id']) for a in areas], description='Área:')
codigo_widget = widgets.Text(value='DEMO-004', description='Código:')
tipo_widget = widgets.Dropdown(options=['vacaciones', 'permiso'], description='Tipo:')
inicio_widget = widgets.DatePicker(description='Desde:')
fin_widget = widgets.DatePicker(description='Hasta:')
guardar_widget = widgets.Button(description='Registrar', button_style='primary')
salida_widget = widgets.Output()

def registrar(_):
    with salida_widget:
        clear_output()
        codigo = codigo_widget.value.strip().upper()
        if not codigo.startswith('DEMO-'):
            print('Usa un código ficticio que comience con DEMO-.')
            return
        if inicio_widget.value is None or fin_widget.value is None or fin_widget.value < inicio_widget.value:
            print('Selecciona fechas válidas; la fecha final debe ser igual o posterior a la inicial.')
            return
        cur.execute('''INSERT INTO solicitudes
            (codigo_trabajador, area_id, tipo, fecha_inicio, fecha_fin, estado, observacion_demo)
            VALUES (%s, %s, %s, %s, %s, 'pendiente', 'Registro ficticio desde Colab')''',
            (codigo, area_widget.value, tipo_widget.value, inicio_widget.value, fin_widget.value))
        conn.commit()
        print('Solicitud ficticia registrada.')

guardar_widget.on_click(registrar)
display(widgets.VBox([codigo_widget, area_widget, tipo_widget, inicio_widget, fin_widget, guardar_widget, salida_widget]))

## 5. Cerrar la conexión

Ejecuta esta celda al terminar. Para volver a iniciar la base, repite desde la sección 1.

In [ ]:
cur.close()
conn.close()
print('Conexión cerrada.')